In [11]:
import pandas as pd

data = pd.read_csv("../chronic_dataset/chronic_kidney_disease_corrected.csv")

print("Dataset Shape:", data.shape)
print("\nMissing Values in Each Column:")
print(data.isnull().sum())

Dataset Shape: (400, 25)

Missing Values in Each Column:
age        9
bp        12
sg        47
al        46
su        49
rbc      152
pc        65
pcc        4
ba         4
bgr       44
bu        19
sc        17
sod       87
pot       88
hemo      52
pcv       71
wbcc     106
rbcc     131
htn        2
dm         3
cad        2
appet      1
pe         1
ane        1
class      0
dtype: int64


## mean median mode

In [12]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# STEP 1: Load Original Dataset
data = pd.read_csv("../chronic_dataset/chronic_kidney_disease_corrected.csv")

# Clean categorical text
for column in data.select_dtypes(include=["object", "str"]).columns:
    data[column] = data[column].str.strip()

# STEP 2: Separate Features and Target
X = data.drop(columns=["class"])
y = data["class"]

# STEP 3: Identify Numerical and Categorical Columns
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "str"]).columns.tolist()

# STEP 4: Split Dataset (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# STEP 5: Median Imputation for Numerical Columns
median_imputer = SimpleImputer(strategy="median")

X_train[numerical_columns] = median_imputer.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = median_imputer.transform(
    X_test[numerical_columns]
)

# STEP 6: Mode Imputation for Categorical Columns
mode_imputer = SimpleImputer(strategy="most_frequent")

X_train[categorical_columns] = mode_imputer.fit_transform(
    X_train[categorical_columns]
)

X_test[categorical_columns] = mode_imputer.transform(
    X_test[categorical_columns]
)

# STEP 7: Combine Features and Target
train_data = X_train.copy()
train_data["class"] = y_train

test_data = X_test.copy()
test_data["class"] = y_test

# STEP 8: Create Output Folder
output_folder = Path("../cleaned_datasets")
output_folder.mkdir(parents=True, exist_ok=True)

# STEP 9: Save Training and Testing Datasets
train_path = output_folder / "median_mode_train_clean.csv"
test_path = output_folder / "median_mode_test_clean.csv"

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

# STEP 10: Verify Results
print("Training Dataset Shape:", train_data.shape)
print("Testing Dataset Shape:", test_data.shape)

print("\nTraining Missing Values:", train_data.isnull().sum().sum())
print("Testing Missing Values:", test_data.isnull().sum().sum())

print("\nTraining Class Distribution:")
print(train_data["class"].value_counts())

print("\nTesting Class Distribution:")
print(test_data["class"].value_counts())

print("\nSaved Training File:", train_path)
print("Saved Testing File:", test_path)

Training Dataset Shape: (320, 25)
Testing Dataset Shape: (80, 25)

Training Missing Values: 0
Testing Missing Values: 0

Training Class Distribution:
class
ckd       200
notckd    120
Name: count, dtype: int64

Testing Class Distribution:
class
ckd       50
notckd    30
Name: count, dtype: int64

Saved Training File: ..\cleaned_datasets\median_mode_train_clean.csv
Saved Testing File: ..\cleaned_datasets\median_mode_test_clean.csv


## knn

In [13]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

# STEP 1: Load Original Dataset
data = pd.read_csv("../chronic_dataset/chronic_kidney_disease_corrected.csv")

# Clean categorical text
for column in data.select_dtypes(include=["object", "str"]).columns:
    data[column] = data[column].str.strip()

# STEP 2: Separate Features and Target
X = data.drop(columns=["class"])
y = data["class"]

# STEP 3: Identify Numerical and Categorical Columns
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "str"]).columns.tolist()

# STEP 4: Split Dataset (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# STEP 5: Scale Numerical Features
scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

# STEP 6: KNN Imputation for Numerical Features
knn_imputer = KNNImputer(n_neighbors=5)

X_train[numerical_columns] = knn_imputer.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = knn_imputer.transform(
    X_test[numerical_columns]
)

# STEP 7: Restore Original Numerical Scale
X_train[numerical_columns] = scaler.inverse_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.inverse_transform(
    X_test[numerical_columns]
)

# STEP 8: Mode Imputation for Categorical Features
mode_imputer = SimpleImputer(strategy="most_frequent")

X_train[categorical_columns] = mode_imputer.fit_transform(
    X_train[categorical_columns]
)

X_test[categorical_columns] = mode_imputer.transform(
    X_test[categorical_columns]
)

# STEP 9: Combine Features and Target
train_data = X_train.copy()
train_data["class"] = y_train

test_data = X_test.copy()
test_data["class"] = y_test

# STEP 10: Save Cleaned Datasets
output_folder = Path("../cleaned_datasets")
output_folder.mkdir(parents=True, exist_ok=True)

train_path = output_folder / "knn_train_clean.csv"
test_path = output_folder / "knn_test_clean.csv"

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

# STEP 11: Verify Results
print("Training Dataset Shape:", train_data.shape)
print("Testing Dataset Shape:", test_data.shape)

print("\nTraining Missing Values:", train_data.isnull().sum().sum())
print("Testing Missing Values:", test_data.isnull().sum().sum())

print("\nTraining Class Distribution:")
print(train_data["class"].value_counts())

print("\nTesting Class Distribution:")
print(test_data["class"].value_counts())

print("\nSaved Training File:", train_path)
print("Saved Testing File:", test_path)

Training Dataset Shape: (320, 25)
Testing Dataset Shape: (80, 25)

Training Missing Values: 0
Testing Missing Values: 0

Training Class Distribution:
class
ckd       200
notckd    120
Name: count, dtype: int64

Testing Class Distribution:
class
ckd       50
notckd    30
Name: count, dtype: int64

Saved Training File: ..\cleaned_datasets\knn_train_clean.csv
Saved Testing File: ..\cleaned_datasets\knn_test_clean.csv


## iterative

In [14]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler

# STEP 1: Load Original Dataset
data = pd.read_csv("../chronic_dataset/chronic_kidney_disease_corrected.csv")

# Clean categorical text
for column in data.select_dtypes(include=["object", "str"]).columns:
    data[column] = data[column].str.strip()

# STEP 2: Separate Features and Target
X = data.drop(columns=["class"])
y = data["class"].copy()

# STEP 3: Identify Numerical and Categorical Columns
numerical_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "str"]).columns.tolist()

# STEP 4: Split Dataset (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# STEP 5: Scale Numerical Features
scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

# STEP 6: Iterative Imputation for Numerical Features
iterative_imputer = IterativeImputer(
    max_iter=30,
    random_state=42
)

X_train[numerical_columns] = iterative_imputer.fit_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = iterative_imputer.transform(
    X_test[numerical_columns]
)

# STEP 7: Restore Original Numerical Scale
X_train[numerical_columns] = scaler.inverse_transform(
    X_train[numerical_columns]
)

X_test[numerical_columns] = scaler.inverse_transform(
    X_test[numerical_columns]
)

# STEP 8: Mode Imputation for Categorical Features
mode_imputer = SimpleImputer(strategy="most_frequent")

X_train[categorical_columns] = mode_imputer.fit_transform(
    X_train[categorical_columns]
)

X_test[categorical_columns] = mode_imputer.transform(
    X_test[categorical_columns]
)

# STEP 9: Combine Features and Target
train_data = X_train.copy()
train_data["class"] = y_train

test_data = X_test.copy()
test_data["class"] = y_test

# STEP 10: Save Cleaned Datasets
output_folder = Path("../cleaned_datasets")
output_folder.mkdir(parents=True, exist_ok=True)

train_path = output_folder / "iterative_train_clean.csv"
test_path = output_folder / "iterative_test_clean.csv"

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

# STEP 11: Verify Results
print("Training Dataset Shape:", train_data.shape)
print("Testing Dataset Shape:", test_data.shape)

print("\nTraining Missing Values:", train_data.isnull().sum().sum())
print("Testing Missing Values:", test_data.isnull().sum().sum())

print("\nTraining Class Distribution:")
print(train_data["class"].value_counts())

print("\nTesting Class Distribution:")
print(test_data["class"].value_counts())

print("\nSaved Training File:", train_path)
print("Saved Testing File:", test_path)

Training Dataset Shape: (320, 25)
Testing Dataset Shape: (80, 25)

Training Missing Values: 0
Testing Missing Values: 0

Training Class Distribution:
class
ckd       200
notckd    120
Name: count, dtype: int64

Testing Class Distribution:
class
ckd       50
notckd    30
Name: count, dtype: int64

Saved Training File: ..\cleaned_datasets\iterative_train_clean.csv
Saved Testing File: ..\cleaned_datasets\iterative_test_clean.csv
